# Limpieza del conjunto de datos

Este notebook aplica el plan de limpieza descrito en el documento del proyecto, sobre los hallazgos confirmados en `01_diagnostico.ipynb`.

Principios seguidos:
- No se elimina ningún registro sin documentar la razón.
- Toda transformación queda registrada en la tabla de transformaciones del documento.
- Los puntos que idealmente requerirían confirmación con el MINEDUC se marcan explícitamente como **NOTA** y se resuelven con la decisión más razonable disponible, sin bloquear el proceso.
- Al final se genera `data/processed/centros_educativos_gt_limpio.csv`.


In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from difflib import SequenceMatcher

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Acumulador para la tabla de transformaciones (punto 6 de la guía)
LOG_TRANSFORMACIONES = []

def registrar(variable, problema, transformacion, registros_afectados, justificacion):
    LOG_TRANSFORMACIONES.append({
        "Variable": variable,
        "Problema detectado": problema,
        "Transformación": transformacion,
        "Registros afectados": registros_afectados,
        "Justificación": justificacion,
    })


## Carga de datos crudos

In [2]:
df = pd.read_csv("../data/processed/centros_educativos_gt.csv", dtype=str, encoding="utf-8", header=1)
print("Filas:", len(df), " Columnas:", len(df.columns))
df.head(3)


Filas: 11889  Columnas: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


## 1. Filas de encabezado duplicado (hallazgo estructural)

Al unir los CSV por departamento (`unir_datos.py`), la fila de encabezado de cada archivo individual quedó
incluida como una fila de datos más. Se identifican y eliminan las filas donde `CODIGO == "CODIGO"`
(equivalente a que cada columna contenga literalmente su propio nombre).


In [3]:
filas_fantasma = df["CODIGO"] == "CODIGO"
n_fantasma = int(filas_fantasma.sum())
print(f"Filas fantasma detectadas: {n_fantasma}")

df = df[~filas_fantasma].copy()
df = df.reset_index(drop=True)
print("Filas tras eliminar encabezados duplicados:", len(df))

registrar(
    "Todas (fila completa)",
    "Filas de encabezado de cada CSV departamental quedaron incluidas como datos al concatenar (pipeline unir_datos.py).",
    f"Se eliminaron {n_fantasma} filas donde CODIGO == 'CODIGO'.",
    n_fantasma,
    "No son establecimientos reales; son un artefacto del proceso de unión. Se identificaron por comparación exacta contra el nombre de columna, criterio confirmado en el diagnóstico (01_diagnostico.ipynb)."
)


Filas fantasma detectadas: 22
Filas tras eliminar encabezados duplicados: 11867


## 2. Valores faltantes y placeholders

Se unifican a `NA` real los distintos "placeholders" de dato faltante detectados en el diagnóstico:
cadenas vacías, espacios en blanco, secuencias de solo símbolos (`-`, `--`, `...`) y frases como
`SIN DATO`, `NO TIENE`, `NINGUNO`, `N/A`, `NULL`, etc.


In [4]:
PLACEHOLDERS_TEXTO = {
    "N/A", "NA", "S/N", "S/D", "SIN DATO", "SIN DATOS", "NO TIENE", "NINGUNO",
    "NULO", "NULL", "VACANTE", "NO APLICA", ".",
}
PATRON_SOLO_SIMBOLOS = re.compile(r"^[\-\.,_\s]+$")

def es_placeholder(valor):
    if pd.isna(valor):
        return True
    v = str(valor).strip()
    if v == "":
        return True
    if PATRON_SOLO_SIMBOLOS.match(v):
        return True
    if v.upper() in PLACEHOLDERS_TEXTO:
        return True
    return False

COLUMNAS_TEXTO_LIBRE = ["DISTRITO", "ESTABLECIMIENTO", "DIRECCION", "TELEFONO", "SUPERVISOR", "DIRECTOR"]

for col in COLUMNAS_TEXTO_LIBRE:
    mask = df[col].apply(es_placeholder) & df[col].notna()
    n = int(mask.sum())
    if n > 0:
        df.loc[mask, col] = np.nan
        registrar(
            col,
            "Placeholders de dato faltante (espacios, símbolos sueltos, 'SIN DATO', 'N/A', etc.) no reconocidos como NA por pandas.",
            f"Se convirtieron {n} valores placeholder a NA explícito.",
            n,
            "Unifica el conteo real de valores faltantes; ver nota sobre posibles significados administrativos distintos en el documento"
        )

resumen_na = pd.DataFrame({
    "n_faltantes": df.isna().sum(),
    "pct_faltantes": (df.isna().sum() / len(df) * 100).round(2),
}).sort_values("pct_faltantes", ascending=False)
resumen_na[resumen_na["n_faltantes"] > 0]


,n_faltantes,pct_faltantes
DIRECTOR,2143,18.06
TELEFONO,946,7.97
SUPERVISOR,538,4.53
DISTRITO,532,4.48
DIRECCION,89,0.75
ESTABLECIMIENTO,5,0.04


## 3. Normalización de texto

Trim, colapso de espacios múltiples y normalización Unicode (NFC) en todas las columnas de texto,
para evitar caracteres compuestos duplicados (ej. tilde como carácter combinante vs. carácter precompuesto).


In [5]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return valor
    v = unicodedata.normalize("NFC", str(valor))
    v = v.replace("\xa0", " ")
    v = re.sub(r"\s+", " ", v).strip()
    return v

columnas_texto = [c for c in df.columns]
afectadas_espacios = 0
for col in columnas_texto:
    original = df[col]
    normalizado = original.apply(normalizar_texto)
    cambio = (original.fillna("") != normalizado.fillna(""))
    afectadas_espacios += int(cambio.sum())
    df[col] = normalizado

registrar(
    "Todas (texto)",
    "Espacios al inicio/fin, espacios múltiples internos y formas Unicode no normalizadas (NFC).",
    "Trim + colapso de espacios múltiples + normalización Unicode NFC aplicados a todas las columnas.",
    afectadas_espacios,
    "Estandariza la representación de texto sin alterar el contenido semántico; requerido por el punto 5c de la guía."
)
print("Celdas de texto modificadas por normalización de espacios/Unicode:", afectadas_espacios)


Celdas de texto modificadas por normalización de espacios/Unicode: 0


## 4. `ESTABLECIMIENTO` — comillas embebidas y siglas institucionales

El diagnóstico encontró 2,228 registros con comillas dobles embebidas (ej. `COLEGIO "LA INMACULADA"`)
y 265 registros que terminan en una sigla institucional entre guiones (ej. `-IIAV-`).
Se retiran las comillas (ruido de captura) y se conserva la sigla en el nombre, además de extraerla
como variable derivada `ESTABLECIMIENTO_SIGLA`.


In [6]:
n_comillas = int(df["ESTABLECIMIENTO"].str.contains('"', na=False).sum())
df["ESTABLECIMIENTO"] = df["ESTABLECIMIENTO"].str.replace('"', "", regex=False)
df["ESTABLECIMIENTO"] = df["ESTABLECIMIENTO"].apply(normalizar_texto)

registrar(
    "ESTABLECIMIENTO",
    "Comillas dobles embebidas en el nombre del establecimiento (ruido de captura del portal).",
    f"Se eliminó el carácter de comillas dobles en {n_comillas} registros.",
    n_comillas,
    "Las comillas no aportan significado y dificultan comparaciones/matching de nombres."
)

patron_sigla = re.compile(r"-(\w{2,10})-\s*$")
siglas = df["ESTABLECIMIENTO"].str.extract(patron_sigla)[0]
df["ESTABLECIMIENTO_SIGLA"] = siglas
n_siglas = int(siglas.notna().sum())

registrar(
    "ESTABLECIMIENTO_SIGLA (derivada)",
    "El nombre del establecimiento a veces incluye una sigla institucional entre guiones al final.",
    f"Se creó la variable derivada ESTABLECIMIENTO_SIGLA extrayendo el texto entre guiones finales en {n_siglas} registros; el nombre original se conserva intacto.",
    n_siglas,
    "Facilita búsquedas/agrupaciones por sigla sin perder el nombre oficial completo. Ver Libro de Códigos."
)
print("Registros con comillas corregidas:", n_comillas)
print("Registros con sigla extraída:", n_siglas)
df.loc[df["ESTABLECIMIENTO_SIGLA"].notna(), ["ESTABLECIMIENTO", "ESTABLECIMIENTO_SIGLA"]].head(5)


Registros con comillas corregidas: 2228
Registros con sigla extraída: 270


,ESTABLECIMIENTO,ESTABLECIMIENTO_SIGLA
9,INSTITUTO INTERCULTRUAL ALTAVERAPACENCESE -IIAV-,IIAV
30,INSTITUTO MIXTO DE EDUCACION BILINGUE INTERCUL...,IMEBI
55,ENBI CENTRO DE EDUCACIÓN EXTRAESCOLAR -CEEX-,CEEX
67,CENTRO DE EDUCACIÓN EXTRAESCOLAR -CEEX-,CEEX
72,CENTRO DE EDUCACIÓN EXTRAESCOLAR -CEEX-,CEEX


## 5. `DEPARTAMENTO` y `MUNICIPIO` — catálogo oficial y caso "CIUDAD CAPITAL"

`CIUDAD CAPITAL` no es uno de los 22 departamentos oficiales de Guatemala; corresponde al departamento
de Guatemala (zona metropolitana). Para esos registros, `MUNICIPIO` contenía `"ZONA N"` en vez de un
municipio real. Se remapea `DEPARTAMENTO` a `GUATEMALA`, se fija `MUNICIPIO = "GUATEMALA"` y se preserva
la zona en una variable derivada `ZONA`.


In [7]:
CATALOGO_DEPARTAMENTOS = {
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA", "EL PROGRESO",
    "ESCUINTLA", "GUATEMALA", "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA",
    "PETEN", "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ", "SAN MARCOS",
    "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ", "TOTONICAPAN", "ZACAPA",
}

mask_ciudad_capital = df["DEPARTAMENTO"] == "CIUDAD CAPITAL"
n_ciudad_capital = int(mask_ciudad_capital.sum())

df["ZONA"] = pd.Series(np.nan, index=df.index, dtype=object)
zona_extraida = df.loc[mask_ciudad_capital, "MUNICIPIO"].str.extract(r"^ZONA\s*(\d+)$")[0]
df.loc[mask_ciudad_capital, "ZONA"] = zona_extraida

df.loc[mask_ciudad_capital, "MUNICIPIO"] = "GUATEMALA"
df.loc[mask_ciudad_capital, "DEPARTAMENTO"] = "GUATEMALA"

registrar(
    "DEPARTAMENTO / MUNICIPIO",
    "'CIUDAD CAPITAL' no es un departamento oficial; es la zona metropolitana del departamento de Guatemala, y MUNICIPIO contenía 'ZONA N' en vez de un municipio real.",
    f"Se remapeó DEPARTAMENTO 'CIUDAD CAPITAL' -> 'GUATEMALA' y MUNICIPIO -> 'GUATEMALA' en {n_ciudad_capital} registros; el número de zona se preservó en la variable derivada ZONA.",
    n_ciudad_capital,
    "Alinea DEPARTAMENTO con el catálogo oficial de 22 departamentos sin perder la granularidad de zona, que queda disponible en ZONA."
)

dominio_invalido = ~df["DEPARTAMENTO"].isin(CATALOGO_DEPARTAMENTOS)
print("Departamentos fuera de catálogo tras el remapeo:", int(dominio_invalido.sum()))
print("Departamentos en el conjunto:", sorted(df["DEPARTAMENTO"].unique()))
df.loc[df["ZONA"].notna(), ["DEPARTAMENTO", "MUNICIPIO", "ZONA"]].head(5)


Departamentos fuera de catálogo tras el remapeo: 0
Departamentos en el conjunto: ['ALTA VERAPAZ', 'BAJA VERAPAZ', 'CHIMALTENANGO', 'CHIQUIMULA', 'EL PROGRESO', 'ESCUINTLA', 'GUATEMALA', 'HUEHUETENANGO', 'IZABAL', 'JALAPA', 'JUTIAPA', 'PETEN', 'QUETZALTENANGO', 'QUICHE', 'RETALHULEU', 'SACATEPEQUEZ', 'SAN MARCOS', 'SANTA ROSA', 'SOLOLA', 'SUCHITEPEQUEZ', 'TOTONICAPAN', 'ZACAPA']


,DEPARTAMENTO,MUNICIPIO,ZONA
1320,GUATEMALA,GUATEMALA,1
1321,GUATEMALA,GUATEMALA,1
1322,GUATEMALA,GUATEMALA,1
1323,GUATEMALA,GUATEMALA,1
1324,GUATEMALA,GUATEMALA,1


## 6. `DEPARTAMENTAL` — variable de zona de supervisión (no se fusiona con `DEPARTAMENTO`)

`DEPARTAMENTAL` subdivide Guatemala en 4 zonas (`GUATEMALA NORTE/SUR/OCCIDENTE/ORIENTE`) y Quiché en 2
(`QUICHÉ`/`QUICHÉ NORTE`), y usa tildes que `DEPARTAMENTO` no usa. El diagnóstico cuantificó el
"no-match" entre ambas en ~6,116 registros.

**NOTA — requeriría confirmar con el MINEDUC:** no es posible determinar, solo con el dataset, si
`DEPARTAMENTAL` es una división administrativa oficial vigente o una categorización interna del portal.
Se decide **no fusionarla** con `DEPARTAMENTO` (se perdería información real de subdivisión regional);
solo se normaliza su formato de texto.


In [8]:
mismatch = (df["DEPARTAMENTO"].astype(str).str.strip() != df["DEPARTAMENTAL"].astype(str).str.strip()).sum()
print(f"DEPARTAMENTO vs DEPARTAMENTAL no coinciden en {int(mismatch)} registros (post-limpieza).")
print("Valores únicos de DEPARTAMENTAL:", sorted(df["DEPARTAMENTAL"].dropna().unique()))

registrar(
    "DEPARTAMENTAL",
    "No corresponde 1:1 con DEPARTAMENTO (subdivide Guatemala y Quiché en zonas de supervisión) y usa tildes que DEPARTAMENTO no usa.",
    "Se conserva sin fusionar con DEPARTAMENTO; solo se normalizó texto (trim, espacios, Unicode NFC) en el paso 3.",
    int(mismatch),
    "NOTA: requeriría confirmar con el MINEDUC si es una división administrativa oficial. Fusionarla sin esa confirmación arriesga perder información real de zona de supervisión."
)


DEPARTAMENTO vs DEPARTAMENTAL no coinciden en 6094 registros (post-limpieza).
Valores únicos de DEPARTAMENTAL: ['ALTA VERAPAZ', 'BAJA VERAPAZ', 'CHIMALTENANGO', 'CHIQUIMULA', 'EL PROGRESO', 'ESCUINTLA', 'GUATEMALA NORTE', 'GUATEMALA OCCIDENTE', 'GUATEMALA ORIENTE', 'GUATEMALA SUR', 'HUEHUETENANGO', 'IZABAL', 'JALAPA', 'JUTIAPA', 'PETÉN', 'QUETZALTENANGO', 'QUICHÉ', 'QUICHÉ NORTE', 'RETALHULEU', 'SACATEPÉQUEZ', 'SAN MARCOS', 'SANTA ROSA', 'SOLOLÁ', 'SUCHITEPÉQUEZ', 'TOTONICAPÁN', 'ZACAPA']


## 7. `DISTRITO` — múltiples formatos (no se fuerza un único patrón)

Se documentan los tres patrones observados (`DD-DD-NNNN`, `DD-NNN`, `OTRO`) sin normalizar su
estructura interna, ya que no está claro si representan esquemas administrativos distintos.

**NOTA — requeriría confirmar con el MINEDUC:** el significado exacto de cada segmento del código de
distrito y si ambos formatos son válidos o uno de ellos es un error de captura.


In [9]:
def clasifica_distrito(x):
    if pd.isna(x):
        return "FALTANTE"
    if re.match(r"^\d{2}-\d{3}$", x):
        return "DD-NNN"
    if re.match(r"^\d{2}-\d{2}-\d{4}$", x):
        return "DD-DD-NNNN"
    return "OTRO"

df["DISTRITO"] = df["DISTRITO"].apply(normalizar_texto)
patrones_distrito = df["DISTRITO"].apply(clasifica_distrito)
print(patrones_distrito.value_counts())

n_otro = int((patrones_distrito == "OTRO").sum())
df["DISTRITO_FORMATO_ATIPICO"] = patrones_distrito == "OTRO"

registrar(
    "DISTRITO",
    "Conviven al menos dos formatos de codificación (DD-NNN y DD-DD-NNNN) y un residual sin patrón reconocible.",
    f"No se fuerza un único formato (se documentan ambos como válidos). Se marcaron {n_otro} filas con formato atípico en DISTRITO_FORMATO_ATIPICO para revisión manual, sin modificar su valor.",
    n_otro,
    "NOTA: requeriría confirmar con el MINEDUC el significado de cada formato antes de normalizar; se prefiere no alterar el dato original a arriesgar destruir información."
)


DISTRITO
DD-DD-NNNN    6226
DD-NNN        5039
FALTANTE       532
OTRO            70
Name: count, dtype: int64


## 8. `TELEFONO` — extracción de dígitos y números múltiples

Se separan los distintos números que puede contener un mismo campo (unidos por coma, `Y`, `FAX`, `EXT.`,
etc.), se extraen solo dígitos dentro de cada segmento y se acepta como válido cualquier grupo de 7 u 8
dígitos (7 = formato telefónico previo a la renumeración nacional de Guatemala). El primer número válido
queda en `TELEFONO`; los adicionales, en la variable derivada `TELEFONO_ADICIONAL`.


In [10]:
def parse_telefonos(valor):
    if pd.isna(valor):
        return []
    texto = str(valor).upper()
    texto = re.sub(r"\b(Y|FAX|EXT\.?|ESTX\.?)\b", ",", texto)
    partes = re.split(r"[,/;]", texto)
    numeros = []
    for parte in partes:
        digitos = re.sub(r"\D", "", parte)
        if len(digitos) in (7, 8):
            numeros.append(digitos)
        elif len(digitos) > 8:
            for i in range(0, len(digitos), 8):
                bloque = digitos[i:i + 8]
                if len(bloque) >= 7:
                    numeros.append(bloque)
    return numeros

telefono_original = df["TELEFONO"].copy()
listas = telefono_original.apply(parse_telefonos)

df["TELEFONO"] = listas.apply(lambda l: l[0] if l else np.nan)
df["TELEFONO_ADICIONAL"] = listas.apply(lambda l: ";".join(l[1:]) if len(l) > 1 else np.nan)

n_no_parseable = int(((telefono_original.notna()) & (listas.apply(len) == 0)).sum())
n_con_adicional = int(df["TELEFONO_ADICIONAL"].notna().sum())
n_normalizados = int((telefono_original.notna() & listas.apply(len).gt(0)).sum())

registrar(
    "TELEFONO",
    "Formatos heterogéneos: separadores mixtos (guion, coma, espacio, palabras), múltiples números por celda, longitudes de 7 u 8 dígitos.",
    f"Se extrajeron y normalizaron {n_normalizados} valores a solo dígitos; se separaron números adicionales en TELEFONO_ADICIONAL ({n_con_adicional} registros); {n_no_parseable} registros no pudieron normalizarse automáticamente y quedaron como NA.",
    n_normalizados,
    "Deja un único número principal en formato consistente (7-8 dígitos) para validación y reporta explícitamente los residuales no parseables para revisión manual."
)

registrar(
    "TELEFONO_ADICIONAL (derivada)",
    "Un mismo registro puede tener más de un número telefónico (oficina + fax, dos contactos, etc.).",
    f"Se creó la variable derivada TELEFONO_ADICIONAL con los números sobrantes tras quedarse con el primero como TELEFONO principal ({n_con_adicional} registros).",
    n_con_adicional,
    "Evita perder información de contacto adicional sin romper el formato de una única variable TELEFONO principal."
)

print("Números no parseables (quedaron en NA):", n_no_parseable)
print("Ejemplos de valores no parseables:")
print(telefono_original[(telefono_original.notna()) & (listas.apply(len) == 0)].unique()[:10])


Números no parseables (quedaron en NA): 19
Ejemplos de valores no parseables:
<StringArray>
['783928', '230835', '25763,26725 Y 21568', '25763, 26725 Y 21568', '26725,25763 Y 21568', '22067', '736617 Y 714495', '439990 ESTX. 225-250', '3033',
 '225052']
Length: 10, dtype: str


## 9. `SUPERVISOR` y `DIRECTOR` — ya cubiertos por el paso de placeholders

`DIRECTOR` tenía además de nulos explícitos (14.6%) un conjunto grande de placeholders puramente
simbólicos (`"--"`, `"---"`, `.` etc.) y frases (`"SIN DATO"`), ya unificados a `NA` en el paso 2.

**NOTA — requeriría confirmar con el MINEDUC:** si algún placeholder distingue "plaza vacante" de
"dato no reportado"; aquí se trataron todos como equivalentes a "sin dato".


In [11]:
print("DIRECTOR — % faltante tras limpieza:", round(df["DIRECTOR"].isna().mean() * 100, 2))
print("SUPERVISOR — % faltante tras limpieza:", round(df["SUPERVISOR"].isna().mean() * 100, 2))


DIRECTOR — % faltante tras limpieza: 18.06
SUPERVISOR — % faltante tras limpieza: 4.53


## 10. Variables categóricas — tipo de dato y dominio

`NIVEL`, `SECTOR`, `AREA`, `STATUS`, `MODALIDAD`, `JORNADA`, `PLAN` se convierten a tipo `category`
y se valida que su dominio coincida con el observado en el diagnóstico (sin valores nuevos fuera de
catálogo, dado que las filas fantasma ya se eliminaron).


In [12]:
COLUMNAS_CATEGORICAS = ["NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]

for col in COLUMNAS_CATEGORICAS:
    n_valores = df[col].nunique()
    df[col] = df[col].astype("category")
    registrar(
        col,
        "Tipo de dato genérico (texto) para una variable categórica de dominio fijo.",
        f"Convertida a tipo category ({n_valores} categorías tras eliminar filas fantasma).",
        len(df),
        "Refleja correctamente que es una variable categórica de dominio limitado; facilita validaciones de dominio futuras."
    )

df[COLUMNAS_CATEGORICAS].describe()


,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN
count,11867,11867,11867,11867,11867,11867,11867
unique,1,4,3,5,2,6,13
top,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,DOBLE,DIARIO(REGULAR)
freq,11867,9891,9461,6860,11394,3772,7452


## 11. `CODIGO` — validación de formato (llave primaria)

In [13]:
patron_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
codigo_invalido = ~df["CODIGO"].str.match(patron_codigo)
print("CODIGO con formato inválido:", int(codigo_invalido.sum()))
print("CODIGO únicos:", df["CODIGO"].nunique(), "de", len(df), "filas")


CODIGO con formato inválido: 0
CODIGO únicos: 11867 de 11867 filas


## 12. Duplicados exactos (verificación)

Tras eliminar las filas fantasma, se espera 0 duplicados exactos y 0 `CODIGO` repetidos.


In [14]:
print("Filas completamente duplicadas:", int(df.duplicated().sum()))
print("CODIGO duplicados:", int(df["CODIGO"].duplicated().sum()))


Filas completamente duplicadas: 0
CODIGO duplicados: 0


## 13. Duplicados parciales

Se agrupan registros por similitud de nombre (`difflib.SequenceMatcher`, umbral 0.85 — equivalente
basado en librería estándar, sin dependencias externas tipo RapidFuzz/Levenshtein) dentro de la misma
dirección normalizada + municipio + departamento. La mayoría de los grupos resultan ser el mismo
establecimiento físico ofrecido en más de una jornada/plan (cada combinación tiene su propio `CODIGO`),
no errores de captura. Se etiquetan pero **no se elimina ningún registro**.


In [15]:
def normalizar_para_comparar(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().replace('"', "")
    x = re.sub(r"[^\w\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

_est_norm = df["ESTABLECIMIENTO"].apply(normalizar_para_comparar)
_dir_norm = df["DIRECCION"].apply(normalizar_para_comparar)

parent = {i: i for i in df.index}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

grupo_key = pd.DataFrame({"DEPARTAMENTO": df["DEPARTAMENTO"], "MUNICIPIO": df["MUNICIPIO"], "DIR": _dir_norm})
for _, g in grupo_key.groupby(["DEPARTAMENTO", "MUNICIPIO", "DIR"]):
    if g["DIR"].iloc[0] == "" or len(g) < 2 or len(g) > 200:
        continue
    idxs = list(g.index)
    n = len(idxs)
    for i in range(n):
        for j in range(i + 1, n):
            a, b = _est_norm[idxs[i]], _est_norm[idxs[j]]
            if a == b or SequenceMatcher(None, a, b).ratio() >= 0.85:
                union(idxs[i], idxs[j])

grupos = {}
for i in df.index:
    r = find(i)
    grupos.setdefault(r, []).append(i)
grupos = {k: v for k, v in grupos.items() if len(v) > 1}

df["GRUPO_DUPLICADO_ID"] = pd.Series(np.nan, index=df.index, dtype=object)
df["TIPO_DUPLICADO"] = pd.Series(np.nan, index=df.index, dtype=object)

for gid, (root, idxs) in enumerate(grupos.items(), start=1):
    sub = df.loc[idxs, ["JORNADA", "PLAN", "SECTOR"]].astype(str)
    tipo = "posible_error" if sub.drop_duplicates().shape[0] == 1 else "oferta_multiple"
    df.loc[idxs, "GRUPO_DUPLICADO_ID"] = gid
    df.loc[idxs, "TIPO_DUPLICADO"] = tipo

n_grupos = len(grupos)
n_filas_grupo = sum(len(v) for v in grupos.values())
n_posible_error = int((df["TIPO_DUPLICADO"] == "posible_error").sum())
n_oferta_multiple = int((df["TIPO_DUPLICADO"] == "oferta_multiple").sum())

print(f"Grupos de posibles duplicados parciales: {n_grupos}")
print(f"Filas involucradas: {n_filas_grupo}")
print(f"  - Clasificadas como 'oferta_multiple' (distinta jornada/plan/sector, no es error): {n_oferta_multiple}")
print(f"  - Clasificadas como 'posible_error' (mismos jornada/plan/sector, misma dirección): {n_posible_error}")

registrar(
    "GRUPO_DUPLICADO_ID / TIPO_DUPLICADO (derivadas)",
    "Existen registros con nombre muy similar en la misma dirección; la mayoría corresponden a ofertas de distinta jornada/plan del mismo establecimiento, no a errores.",
    f"Se crearon las variables derivadas GRUPO_DUPLICADO_ID y TIPO_DUPLICADO ({n_filas_grupo} filas etiquetadas en {n_grupos} grupos). No se eliminó ningún registro.",
    n_filas_grupo,
    "Cumple el requisito de detectar duplicados parciales sin depurarlos automáticamente; deja trazabilidad para revisión manual posterior, priorizando los 'posible_error'."
)


Grupos de posibles duplicados parciales: 1988
Filas involucradas: 5532
  - Clasificadas como 'oferta_multiple' (distinta jornada/plan/sector, no es error): 4934
  - Clasificadas como 'posible_error' (mismos jornada/plan/sector, misma dirección): 598


In [16]:
print("Ejemplos de grupos 'posible_error' (misma jornada/plan/sector, revisar manualmente):")
ejemplo_ids = df.loc[df["TIPO_DUPLICADO"] == "posible_error", "GRUPO_DUPLICADO_ID"].dropna().unique()[:3]
for gid in ejemplo_ids:
    display(df.loc[df["GRUPO_DUPLICADO_ID"] == gid, ["CODIGO", "ESTABLECIMIENTO", "DIRECCION", "TELEFONO", "JORNADA", "PLAN", "SECTOR"]])


Ejemplos de grupos 'posible_error' (misma jornada/plan/sector, revisar manualmente):


,CODIGO,ESTABLECIMIENTO,DIRECCION,TELEFONO,JORNADA,PLAN,SECTOR
18,16-01-0559-46,INSTITUTO NORMAL PRIMARIA BILINGUE INTERCULTUR...,3A. AVE. 6-23 ZONA 11,54457786,VESPERTINA,DIARIO(REGULAR),OFICIAL
121,16-01-8849-46,INSTITUTO NORMAL PRIMARIA BILINGUE INTERCULTUR...,3A. AVE. 6-23 ZONA 11,79521468,VESPERTINA,DIARIO(REGULAR),OFICIAL


,CODIGO,ESTABLECIMIENTO,DIRECCION,TELEFONO,JORNADA,PLAN,SECTOR
27,16-01-0587-46,LICEO MIXTO EN COMPUTACION DEL NORTE,"1A. CALLE Y 2A. AVENIDA LOTE 204, BARRIO EL CE...",48633323,DOBLE,FIN DE SEMANA,PRIVADO
28,16-01-0588-46,LICEO MIXTO EN COMPUTACION DEL NORTE,"1A. CALLE Y 2A. AVENIDA LOTE 204, BARRIO EL CE...",48633323,DOBLE,FIN DE SEMANA,PRIVADO


,CODIGO,ESTABLECIMIENTO,DIRECCION,TELEFONO,JORNADA,PLAN,SECTOR
269,16-09-0007-46,CENTRO EDUCATIVO NUEVA VIDA,8A. CALLE 4-47 ZONA 4,36369056,VESPERTINA,DIARIO(REGULAR),PRIVADO
304,16-09-0870-46,CENTRO EDUCATIVO NUEVA VIDA,"8A. CALLE 4-47, ZONA 4",36369056,VESPERTINA,DIARIO(REGULAR),PRIVADO


In [17]:
print("Ejemplo de grupo 'oferta_multiple' (mismo establecimiento, distinta jornada/plan -> NO es un error):")
ejemplo_ids2 = df.loc[df["TIPO_DUPLICADO"] == "oferta_multiple", "GRUPO_DUPLICADO_ID"].dropna().unique()[:1]
for gid in ejemplo_ids2:
    display(df.loc[df["GRUPO_DUPLICADO_ID"] == gid, ["CODIGO", "ESTABLECIMIENTO", "DIRECCION", "TELEFONO", "JORNADA", "PLAN", "SECTOR"]])


Ejemplo de grupo 'oferta_multiple' (mismo establecimiento, distinta jornada/plan -> NO es un error):


,CODIGO,ESTABLECIMIENTO,DIRECCION,TELEFONO,JORNADA,PLAN,SECTOR
4,16-01-0141-46,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MATUTINA,DIARIO(REGULAR),OFICIAL
25,16-01-0574-46,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A. CALLE 11-10 ZONA 2,78674618,VESPERTINA,DIARIO(REGULAR),OFICIAL


## 14. Consistencia entre variables — resumen final

Chequeos de coherencia entre variables tras la limpieza.


In [18]:
chequeos = pd.DataFrame({
    "Chequeo": [
        "DEPARTAMENTO fuera del catálogo oficial de 22 departamentos",
        "DEPARTAMENTO vs DEPARTAMENTAL no coinciden (ver NOTA sección 6)",
        "CODIGO con formato inválido",
        "CODIGO duplicado",
        "Filas completamente duplicadas",
        "DISTRITO con formato atípico (ver NOTA sección 7)",
    ],
    "Cantidad": [
        int((~df["DEPARTAMENTO"].isin(CATALOGO_DEPARTAMENTOS)).sum()),
        int(mismatch),
        int(codigo_invalido.sum()),
        int(df["CODIGO"].duplicated().sum()),
        int(df.duplicated().sum()),
        int(df["DISTRITO_FORMATO_ATIPICO"].sum()),
    ],
})
chequeos


,Chequeo,Cantidad
0,DEPARTAMENTO fuera del catálogo oficial de 22 ...,0
1,DEPARTAMENTO vs DEPARTAMENTAL no coinciden (ve...,6094
2,CODIGO con formato inválido,0
3,CODIGO duplicado,0
4,Filas completamente duplicadas,0
5,DISTRITO con formato atípico (ver NOTA sección 7),70


## 15. Registro de transformaciones

Tabla completa de todas las transformaciones aplicadas (punto 6 de la guía).


In [19]:
tabla_transformaciones = pd.DataFrame(LOG_TRANSFORMACIONES)
tabla_transformaciones.to_csv("../transformations/registro_transformaciones.csv", index=False)
tabla_transformaciones


,Variable,Problema detectado,Transformación,Registros afectados,Justificación
0,Todas (fila completa),Filas de encabezado de cada CSV departamental ...,Se eliminaron 22 filas donde CODIGO == 'CODIGO'.,22,No son establecimientos reales; son un artefac...
1,DIRECCION,"Placeholders de dato faltante (espacios, símbo...",Se convirtieron 13 valores placeholder a NA ex...,13,Unifica el conteo real de valores faltantes; v...
2,SUPERVISOR,"Placeholders de dato faltante (espacios, símbo...",Se convirtieron 3 valores placeholder a NA exp...,3,Unifica el conteo real de valores faltantes; v...
3,DIRECTOR,"Placeholders de dato faltante (espacios, símbo...",Se convirtieron 411 valores placeholder a NA e...,411,Unifica el conteo real de valores faltantes; v...
4,Todas (texto),"Espacios al inicio/fin, espacios múltiples int...",Trim + colapso de espacios múltiples + normali...,0,Estandariza la representación de texto sin alt...
5,ESTABLECIMIENTO,Comillas dobles embebidas en el nombre del est...,Se eliminó el carácter de comillas dobles en 2...,2228,Las comillas no aportan significado y dificult...
6,ESTABLECIMIENTO_SIGLA (derivada),El nombre del establecimiento a veces incluye ...,Se creó la variable derivada ESTABLECIMIENTO_S...,270,Facilita búsquedas/agrupaciones por sigla sin ...
7,DEPARTAMENTO / MUNICIPIO,'CIUDAD CAPITAL' no es un departamento oficial...,Se remapeó DEPARTAMENTO 'CIUDAD CAPITAL' -> 'G...,2161,Alinea DEPARTAMENTO con el catálogo oficial de...
8,DEPARTAMENTAL,No corresponde 1:1 con DEPARTAMENTO (subdivide...,Se conserva sin fusionar con DEPARTAMENTO; sol...,6094,NOTA: requeriría confirmar con el MINEDUC si e...
9,DISTRITO,Conviven al menos dos formatos de codificación...,No se fuerza un único formato (se documentan a...,70,NOTA: requeriría confirmar con el MINEDUC el s...


## 16. Exportar el conjunto de datos limpio

In [20]:
SALIDA = "../data/processed/centros_educativos_gt_limpio.csv"
df.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado: {SALIDA}")
print("Filas:", len(df), " Columnas:", len(df.columns))
print("Columnas finales:", list(df.columns))


Guardado: ../data/processed/centros_educativos_gt_limpio.csv
Filas: 11867  Columnas: 23
Columnas finales: ['CODIGO', 'DISTRITO', 'DEPARTAMENTO', 'MUNICIPIO', 'ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR', 'NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTAL', 'ESTABLECIMIENTO_SIGLA', 'ZONA', 'DISTRITO_FORMATO_ATIPICO', 'TELEFONO_ADICIONAL', 'GRUPO_DUPLICADO_ID', 'TIPO_DUPLICADO']
